In [0]:
!pip install -r ../requirements.txt
%restart_python

In [0]:
import pandas as pd
import uuid

CATEGORIES_PATH = "/Volumes/abs_data/default/raw_volume/clinton_email_categories.csv"
df = pd.read_csv(CATEGORIES_PATH)

df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

df['id'] = [str(uuid.uuid4()) for _ in range(len(df))]
display(df.head())


In [0]:
from shared.llm_utils import get_embedding

print(get_embedding("where is my cow?"))

In [0]:
df['embedding'] = df.apply(lambda row: get_embedding(f"{row['category']}: {row['description']}"), axis=1)

In [0]:
display(df.head(20))

In [0]:
from shared.catalog_manager import CatalogManager

from pyspark.sql.types import StructType, StructField, StringType, FloatType, ArrayType

category_schema = StructType([
    StructField("id", StringType(), True),
    StructField("category", StringType(), True),
    StructField("description", StringType(), True),
    StructField("embedding", ArrayType(FloatType()), True),
])

catalog_manager = CatalogManager(fqn="clinton_emails.silver.email_categories", schema=category_schema)
catalog_manager.commit_pandas_dataframe(df)
